In [2]:
import numpy as np
from bvh import Bvh

def rotate_bvh(input_bvh, output_bvh, rotation_deg=90, axis='y'):
    with open(input_bvh, 'r') as f:
        bvh = Bvh(f.read())

    rotation_rad = np.radians(rotation_deg)

    if axis == 'y':
        R = np.array([
            [np.cos(rotation_rad), 0, np.sin(rotation_rad)],
            [0, 1, 0],
            [-np.sin(rotation_rad), 0, np.cos(rotation_rad)]
        ])
    elif axis == 'x':
        R = np.array([
            [1, 0, 0],
            [0, np.cos(rotation_rad), -np.sin(rotation_rad)],
            [0, np.sin(rotation_rad), np.cos(rotation_rad)]
        ])
    elif axis == 'z':
        R = np.array([
            [np.cos(rotation_rad), -np.sin(rotation_rad), 0],
            [np.sin(rotation_rad), np.cos(rotation_rad), 0],
            [0, 0, 1]
        ])
    else:
        raise ValueError("Axis must be one of ['x', 'y', 'z'].")

    # ✅ frames là list các list số (float)
    frames = np.array(bvh.frames, dtype=float)

    # Xoay root translation (3 giá trị đầu tiên mỗi frame)
    root_pos = frames[:, :3]
    frames[:, :3] = root_pos @ R.T

    # Ghi lại file mới
    with open(output_bvh, 'w') as f:
        f.write(str(bvh))
        f.write("\nMOTION\n")
        f.write(f"Frames: {bvh.nframes}\n")
        f.write(f"Frame Time: {bvh.frame_time}\n")
        for frame in frames:
            f.write(" ".join(f"{v:.6f}" for v in frame) + "\n")

    print(f"✅ Rotated BVH saved to {output_bvh}")


In [3]:
rotate_bvh("/home/anhndt/animating_image/external/cmu-mocap/data/138/138_16.bvh",
           "/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/speaking.bvh",
           rotation_deg=90,
           axis="y")

✅ Rotated BVH saved to /home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/speaking.bvh


In [8]:
def skip_bvh_frames(input_path: str, output_path: str, step: int = 2):
    """
    Skip frames in a BVH file by keeping only every 'step' frame.
    Adjusts both frame count and frame time.
    """
    hierarchy_lines = []
    motion_lines = []
    in_motion = False

    with open(input_path, "r") as f:
        for line in f:
            if line.strip().startswith("MOTION"):
                in_motion = True
                hierarchy_lines.append(line)
                continue

            if not in_motion:
                hierarchy_lines.append(line)
            else:
                motion_lines.append(line)

    # Extract meta
    original_frames = int(motion_lines[0].split(":")[1].strip())
    old_frame_time = float(motion_lines[1].split(":")[1].strip())

    # Frame data
    frame_data = motion_lines[2:]

    # Skip frames
    new_frame_data = frame_data[::step]
    new_frame_count = len(new_frame_data)

    # Compute new frame time
    new_frame_time = old_frame_time * step

    # Build output
    output_lines = []
    output_lines.extend(hierarchy_lines)
    output_lines.append(f"Frames: {new_frame_count}\n")
    output_lines.append(f"Frame Time: {new_frame_time:.6f}\n")
    output_lines.extend(new_frame_data)

    with open(output_path, "w") as f:
        f.writelines(output_lines)

    return output_path


In [14]:
skip_bvh_frames(
    input_path="/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/standing.bvh",
    output_path="/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/standing_skip_20.bvh",
    step=20  # giữ mỗi 2 frame
)


'/home/anhndt/animating_image/external/AnimatedDrawings/examples/bvh/custom/standing_skip_20.bvh'